# GWM-RNN Training on Kaggle - FB15k-237 Dataset

Train the lightweight GWM-RNN model for Knowledge Graph Completion on FB15k-237.

**Dataset: FB15k-237**
- ~14,500 entities (from Freebase KB)
- 237 relation types
- ~272k training triples
- **Task**: Given `(head, relation, ?)`, predict the tail entity

**Model Advantages:**
- 🚀 **Fast**: 100x faster than LLM-based approaches
- 💾 **Lightweight**: ~5-15M parameters (vs 3-8B for LLMs)
- 💰 **Efficient**: Trains on consumer GPUs in 2-3 hours
- 📊 **Competitive**: Achieves strong performance (MRR ~0.30+)

**New Features:**
- ✅ **Fixed Negative Sampling**: All models use identical negatives for fair comparison
- ✅ **Automatic Progress Tracking**: CSV updates after each model completes
- ✅ **Training Curves**: Automatic visualization generation
- ✅ **Prediction Saving**: Store model predictions for later analysis

**Training Time:** ~2-3 hours for all experiments on P100 GPU

---

## 1. Install Dependencies

In [ ]:
import os
import sys

# Check environment
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Running on Kaggle: {IS_KAGGLE}")

if IS_KAGGLE:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("✓ Environment setup complete")

In [ ]:
# Install required packages
!pip install -q torch scikit-learn tqdm matplotlib seaborn

print("✓ All dependencies installed")

## 2. Configuration

Configure paths and training parameters for FB15k-237. We'll train multiple configurations:
- **3 Pooling Methods**: last, mean, max
- **Multiple Hyperparameter Sets**: Different hidden dimensions and learning rates
- **2 Loss Functions**: InfoNCE (contrastive) and Margin Ranking

In [ ]:
# ==============================================================================
# DATA PATHS CONFIGURATION
# ==============================================================================
if IS_KAGGLE:
    # Kaggle input paths for FB15k-237 dataset
    DATA_DIR = '/kaggle/input/gwm-rnn-kg-fb15k237'  # Your processed FB15k-237 data
    OUTPUT_BASE_DIR = '/kaggle/working/experiments'
else:
    # Local paths
    DATA_DIR = 'D:/NLP research/Code/graph-world-models/GWM/data/fb15k-237/processed/relation-prediction'
    OUTPUT_BASE_DIR = './trained/gwm-rnn/fb15k-237/experiments'

# ==============================================================================
# POOLING METHODS TO TEST
# ==============================================================================
POOLING_METHODS = ['last', 'mean', 'max']

# ==============================================================================
# HYPERPARAMETER CONFIGURATIONS
# ==============================================================================
# Define multiple configurations to test
HYPERPARAMETER_SETS = [
    # Standard configuration (balanced performance)
    {
        'name': 'standard',
        'hidden_dim': 512,
        'num_lstm_layers': 2,
        'dropout': 0.1,
        'learning_rate': 1e-3,
        'batch_size': 512,
        'num_negatives': 10,
        'loss': 'infonce',
        'temperature': 0.07,
        'use_in_batch_negatives': False,
        'description': 'Standard config with random negatives'
    },
    # In-batch negatives (most efficient)
    {
        'name': 'in-batch',
        'hidden_dim': 512,
        'num_lstm_layers': 2,
        'dropout': 0.1,
        'learning_rate': 1e-3,
        'batch_size': 512,
        'num_negatives': 0,  # Not used with in-batch negatives
        'loss': 'infonce',
        'temperature': 0.07,
        'use_in_batch_negatives': True,
        'description': 'InfoNCE with in-batch negatives (511 negatives per sample)'
    },
    # Larger model (more capacity)
    {
        'name': 'large',
        'hidden_dim': 768,
        'num_lstm_layers': 2,
        'dropout': 0.1,
        'learning_rate': 5e-4,
        'batch_size': 1024,
        'num_negatives': 0,
        'loss': 'infonce',
        'temperature': 0.05,
        'use_in_batch_negatives': True,
        'description': 'Larger hidden dimension, more negatives'
    },
    # Deeper model
    {
        'name': 'deep',
        'hidden_dim': 512,
        'num_lstm_layers': 3,
        'dropout': 0.2,
        'learning_rate': 8e-4,
        'batch_size': 256,
        'num_negatives': 10,
        'loss': 'infonce',
        'temperature': 0.07,
        'use_in_batch_negatives': False,
        'description': 'Deeper network (3 LSTM layers)'
    },
]

# ==============================================================================
# TRAINING PARAMETERS (FIXED ACROSS ALL EXPERIMENTS)
# ==============================================================================
NUM_EPOCHS = 100
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 10
SCHEDULER_PATIENCE = 5
EVAL_EVERY = 1
SEED = 42
NUM_WORKERS = 2

# ==============================================================================
# EXPERIMENT SELECTION
# ==============================================================================
# Choose which experiments to run
RUN_ALL_CONFIGS = True  # Set True to run all hyperparameter sets
SELECTED_CONFIG = 'standard'  # Which config to use if RUN_ALL_CONFIGS=False

print("="*80)
print(" "*15 + "GWM-RNN EXPERIMENT CONFIGURATION - FB15k-237")
print("="*80)
print(f"\n📊 Dataset: FB15k-237 (Freebase Knowledge Graph)")
print(f"   ~14,500 entities")
print(f"   237 relation types (474 with inverses)")
print(f"   ~545k training triples (with inverse relations)")
print(f"   Task: Knowledge Graph Completion")

print(f"\n📁 Data Directory: {DATA_DIR}")
print(f"📁 Output Base Directory: {OUTPUT_BASE_DIR}")

print(f"\n🔄 Pooling Methods to Test ({len(POOLING_METHODS)}):")
for pooling in POOLING_METHODS:
    print(f"   • {pooling}")

print(f"\n⚙️  Hyperparameter Sets Available ({len(HYPERPARAMETER_SETS)}):")
for i, config in enumerate(HYPERPARAMETER_SETS, 1):
    print(f"   {i}. {config['name']:15s} - {config['description']}")
    neg_info = f"In-batch ({config['batch_size']-1} negatives)" if config.get('use_in_batch_negatives', False) else f"{config['num_negatives']} sampled"
    print(f"      Hidden: {config['hidden_dim']}, Layers: {config['num_lstm_layers']}, "
          f"Loss: {config['loss']}, Negatives: {neg_info}")

if RUN_ALL_CONFIGS:
    print(f"\n🚀 Mode: Running ALL configurations")
    print(f"   Total experiments: {len(POOLING_METHODS)} pooling × {len(HYPERPARAMETER_SETS)} configs = {len(POOLING_METHODS) * len(HYPERPARAMETER_SETS)} experiments")
    print(f"   Expected time: ~6-8 hours on P100 GPU")
else:
    print(f"\n🎯 Mode: Running SELECTED configuration only")
    print(f"   Config: {SELECTED_CONFIG}")
    print(f"   Total experiments: {len(POOLING_METHODS)} pooling × 1 config = {len(POOLING_METHODS)} experiments")
    print(f"   Expected time: ~2-3 hours on P100 GPU")

print(f"\n⏱️  Fixed Parameters:")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Gradient clipping: {MAX_GRAD_NORM}")
print(f"   Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"   Seed: {SEED}")

print(f"\n💡 Note: Evaluation uses filtered ranking (MRR, Hits@K metrics)")
print("="*80)

## 3. Copy Training Files from GitHub

Clone repository and copy training scripts for relation prediction.

In [ ]:
required_files = ['model.py', 'dataset.py', 'inference.py', 'train.py', 'utils.py', 'generate_negatives.py']

if IS_KAGGLE:
    print("="*70)
    print("Cloning GitHub repository...")
    print("="*70)
    
    # Clone your GitHub repo
    GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
    BRANCH = "main"
    
    !git clone {GITHUB_REPO} /kaggle/working/gwm
    %cd /kaggle/working/gwm
    !git checkout {BRANCH}
    !git pull
    %cd ../
    
    # Copy files from repo to working directory
    repo_path = "/kaggle/working/gwm/gwm-rnn/relation-prediction"
    
    print(f"\nCopying files from {repo_path}...")
    for file in required_files:
        !cp {repo_path}/{file} /kaggle/working/
        print(f"✓ Copied {file}")
else:
    print("Running locally - files should be in current directory")

# Verify files exist
import os
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"\n❌ Missing files: {missing_files}")
    raise FileNotFoundError(f"Required files not found: {missing_files}")
else:
    print(f"\n✓ All required files ready: {required_files}")

## 3.5 Generate Fixed Negative Samples

**Important**: Generate fixed negative samples to ensure fair comparison across all model variants.

All models will use the same negative samples, eliminating variance from random sampling.

In [ ]:
import torch
from pathlib import Path

# On Kaggle, save to working directory (input is read-only)
if IS_KAGGLE:
    negatives_output_dir = Path('/kaggle/working')
else:
    negatives_output_dir = Path(DATA_DIR)

negatives_path = negatives_output_dir / 'train_negatives.pt'

# Also check if negatives exist in DATA_DIR (for pre-uploaded datasets)
data_dir_negatives = Path(DATA_DIR) / 'train_negatives.pt'

# Determine max negatives needed across all configs
max_negatives = max(config['num_negatives'] for config in HYPERPARAMETER_SETS)
print(f"ℹ️  Maximum negatives needed: {max_negatives}")

if negatives_path.exists():
    print("=" * 70)
    print("✓ Fixed negatives already exist")
    print("=" * 70)
    print(f"Location: {negatives_path}")
    
    # Load and verify
    train_negatives = torch.load(negatives_path, map_location='cpu')
    print(f"Shape: {train_negatives.shape}")
    print(f"Size: {negatives_path.stat().st_size / (1024**2):.2f} MB")
    
    # Check if we have enough negatives
    if train_negatives.shape[1] < max_negatives:
        print()
        print(f"⚠️  WARNING: Existing negatives have {train_negatives.shape[1]} samples")
        print(f"   but {max_negatives} are needed for some configs.")
        print(f"   Regenerating with {max_negatives} negatives...")
        print("=" * 70)
        
        # Regenerate with more negatives
        !python generate_negatives.py \
            --data_dir {DATA_DIR} \
            --output_dir {negatives_output_dir} \
            --num_negatives {max_negatives} \
            --seed {SEED} \
            --force
    else:
        print()
        print("All experiments will use these identical negative samples.")
        print(f"(Each experiment will use the first N negatives as needed)")
        print("=" * 70)
elif data_dir_negatives.exists():
    print("=" * 70)
    print("✓ Fixed negatives found in dataset")
    print("=" * 70)
    print(f"Location: {data_dir_negatives}")
    
    # Load and check
    train_negatives = torch.load(data_dir_negatives, map_location='cpu')
    print(f"Shape: {train_negatives.shape}")
    
    if train_negatives.shape[1] < max_negatives:
        print()
        print(f"⚠️  WARNING: Existing negatives have {train_negatives.shape[1]} samples")
        print(f"   but {max_negatives} are needed. Regenerating...")
        print("=" * 70)
        
        !python generate_negatives.py \
            --data_dir {DATA_DIR} \
            --output_dir {negatives_output_dir} \
            --num_negatives {max_negatives} \
            --seed {SEED} \
            --force
    else:
        # Copy to working directory for consistency
        import shutil
        shutil.copy(data_dir_negatives, negatives_path)
        print(f"Copied to: {negatives_path}")
        print(f"Size: {negatives_path.stat().st_size / (1024**2):.2f} MB")
        print()
        print("All experiments will use these identical negative samples.")
        print("=" * 70)
else:
    print("=" * 70)
    print("GENERATING FIXED NEGATIVE SAMPLES")
    print("=" * 70)
    print()
    print(f"Generating {max_negatives} negatives per triple for all experiments")
    print(f"(Each config will use the first N as needed)")
    print()
    
    # Run the generation script with output to working directory
    !python generate_negatives.py \
        --data_dir {DATA_DIR} \
        --output_dir {negatives_output_dir} \
        --num_negatives {max_negatives} \
        --seed {SEED}
    
    print()
    print("=" * 70)
    print("GENERATION COMPLETE")
    print("=" * 70)
    print()
    print("✓ All experiments will now use these identical negative samples")
    print("✓ This ensures fair comparison across all model configurations")
    print("=" * 70)


## 4. Run Training Experiments

Train models with all pooling methods and selected hyperparameter configurations on FB15k-237.

**Note**: All models use the same pre-generated negative samples for fair comparison.

In [ ]:
import time
from datetime import datetime
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Determine which configs to run
if RUN_ALL_CONFIGS:
    configs_to_run = HYPERPARAMETER_SETS
else:
    configs_to_run = [c for c in HYPERPARAMETER_SETS if c['name'] == SELECTED_CONFIG]

# Track all experiment results
all_results = []
experiment_start_time = time.time()

print("="*80)
print(" "*20 + "STARTING FB15k-237 EXPERIMENTS")
print("="*80)
print(f"\nTotal experiments to run: {len(POOLING_METHODS)} × {len(configs_to_run)} = {len(POOLING_METHODS) * len(configs_to_run)}")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"⏱️  Expected time: ~{len(POOLING_METHODS) * len(configs_to_run) * 60:.0f}-{len(POOLING_METHODS) * len(configs_to_run) * 90:.0f} minutes")
print("="*80)

experiment_num = 0
total_experiments = len(POOLING_METHODS) * len(configs_to_run)

for config in configs_to_run:
    for pooling in POOLING_METHODS:
        experiment_num += 1
        
        print(f"\n{'='*80}")
        print(f" EXPERIMENT {experiment_num}/{total_experiments}: {config['name'].upper()} + {pooling.upper()}-POOLING (FB15k-237)")
        print(f"{'='*80}")
        
        # Create output directory for this experiment
        output_dir = f"{OUTPUT_BASE_DIR}/{config['name']}/{pooling}-pooling"
        
        # Build training command
        if config['loss'] == 'infonce':
            loss_args = f"--loss infonce --temperature {config.get('temperature', 0.07)}"
            if config.get('use_in_batch_negatives', False):
                loss_args += " --use_in_batch_negatives"
        else:
            loss_args = f"--loss margin --margin {config.get('margin', 1.0)}"
        
        cmd = f"""python train.py \\
            --data_dir {DATA_DIR} \\
            --output_dir {output_dir} \\
            --hidden_dim {config['hidden_dim']} \\
            --num_lstm_layers {config['num_lstm_layers']} \\
            --dropout {config['dropout']} \\
            --pooling {pooling} \\
            --num_epochs {NUM_EPOCHS} \\
            --batch_size {config['batch_size']} \\
            --learning_rate {config['learning_rate']} \\
            --weight_decay {WEIGHT_DECAY} \\
            --max_grad_norm {MAX_GRAD_NORM} \\
            --num_negatives {config['num_negatives']} \\
            {loss_args} \\
            --scheduler_patience {SCHEDULER_PATIENCE} \\
            --early_stopping_patience {EARLY_STOPPING_PATIENCE} \\
            --eval_every {EVAL_EVERY} \\
            --seed {SEED} \\
            --num_workers {NUM_WORKERS}"""
        
        print(f"\n📋 Configuration:")
        print(f"   Dataset: FB15k-237 (~14,500 entities, 237 relations)")
        print(f"   Config: {config['name']} - {config['description']}")
        print(f"   Pooling: {pooling}")
        print(f"   Hidden dim: {config['hidden_dim']}")
        print(f"   LSTM layers: {config['num_lstm_layers']}")
        print(f"   Dropout: {config['dropout']}")
        print(f"   Loss: {config['loss']}")
        if config.get('use_in_batch_negatives', False):
            print(f"   Negatives: In-batch ({config['batch_size']-1} per sample)")
        else:
            print(f"   Negatives: {config['num_negatives']} sampled per sample")
        print(f"   Learning rate: {config['learning_rate']}")
        print(f"   Batch size: {config['batch_size']}")
        print(f"   Output: {output_dir}")
        
        print(f"\n🚀 Starting training...")
        print("-"*80)
        
        # Execute training
        !{cmd}
        
        # Load and store results
        try:
            result_path = Path(output_dir) / "test_results.json"
            history_path = Path(output_dir) / "training_history.json"
            
            if result_path.exists() and history_path.exists():
                with open(result_path) as f:
                    test_results = json.load(f)
                with open(history_path) as f:
                    history = json.load(f)
                
                # Get test metrics
                test_metrics = test_results['test_metrics']
                
                # Calculate training time
                train_time = sum(history['epoch_times'])
                
                # Store result
                all_results.append({
                    'config_name': config['name'],
                    'pooling': pooling,
                    'hidden_dim': config['hidden_dim'],
                    'num_layers': config['num_lstm_layers'],
                    'dropout': config['dropout'],
                    'loss': config['loss'],
                    'learning_rate': config['learning_rate'],
                    'batch_size': config['batch_size'],
                    'num_negatives': config['num_negatives'],
                    'use_in_batch_negatives': config.get('use_in_batch_negatives', False),
                    'test_mrr': test_metrics['MRR'],
                    'test_mr': test_metrics['MR'],
                    'test_hits@1': test_metrics['Hits@1'],
                    'test_hits@3': test_metrics['Hits@3'],
                    'test_hits@10': test_metrics['Hits@10'],
                    'test_hits@50': test_metrics['Hits@50'],
                    'best_val_mrr': test_results['best_val_mrr'],
                    'best_epoch': test_results['best_epoch'],
                    'training_time': train_time,
                    'output_dir': output_dir
                })
                
                print(f"\n✅ Experiment {experiment_num} completed successfully!")
                print(f"   Test MRR: {test_metrics['MRR']:.4f}")
                print(f"   Test Hits@10: {test_metrics['Hits@10']:.4f} ({test_metrics['Hits@10']*100:.2f}%)")
                print(f"   Training time: {train_time:.1f}s ({train_time/60:.1f} min)")
            else:
                print(f"\n⚠️  Results not found for experiment {experiment_num}")
        except Exception as e:
            print(f"\n❌ Error loading results for experiment {experiment_num}: {e}")
        
        print(f"\n{'='*80}\n")

total_time = time.time() - experiment_start_time

print(f"\n{'='*80}")
print(" "*20 + "ALL FB15k-237 EXPERIMENTS COMPLETED")
print(f"{'='*80}")
print(f"Total experiments: {len(all_results)}/{total_experiments}")
print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
if len(all_results) > 0:
    print(f"Average time per experiment: {total_time/len(all_results)/60:.1f} minutes")
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}\n")

## 5. Comprehensive Results Analysis

Analyze and compare all FB15k-237 experiments across pooling methods and hyperparameter sets.

In [ ]:
if len(all_results) > 0:
    # Create DataFrame
    df_results = pd.DataFrame(all_results)
    
    print("="*80)
    print(" "*15 + "FB15k-237 EXPERIMENT RESULTS SUMMARY")
    print("="*80)
    
    # Overall best
    best_idx = df_results['test_mrr'].idxmax()
    best_result = df_results.loc[best_idx]
    
    print(f"\n🏆 BEST OVERALL PERFORMANCE:")
    print(f"   Config: {best_result['config_name']} + {best_result['pooling']}-pooling")
    print(f"   Test MRR: {best_result['test_mrr']:.4f}")
    print(f"   Test Hits@1: {best_result['test_hits@1']:.4f} ({best_result['test_hits@1']*100:.2f}%)")
    print(f"   Test Hits@10: {best_result['test_hits@10']:.4f} ({best_result['test_hits@10']*100:.2f}%)")
    print(f"   Test MR: {best_result['test_mr']:.2f}")
    print(f"   Training time: {best_result['training_time']:.1f}s ({best_result['training_time']/60:.1f} min)")
    
    # Pooling method comparison
    print(f"\n📊 PERFORMANCE BY POOLING METHOD:")
    print("-"*80)
    pooling_summary = df_results.groupby('pooling').agg({
        'test_mrr': ['mean', 'std', 'max'],
        'test_hits@10': ['mean', 'max'],
        'test_mr': ['mean', 'min'],
        'training_time': 'mean'
    }).round(4)
    
    for pooling in POOLING_METHODS:
        if pooling in pooling_summary.index:
            row = pooling_summary.loc[pooling]
            print(f"\n{pooling.upper()}-Pooling:")
            print(f"   Avg MRR: {row[('test_mrr', 'mean')]:.4f} ± {row[('test_mrr', 'std')]:.4f}")
            print(f"   Max MRR: {row[('test_mrr', 'max')]:.4f}")
            print(f"   Avg Hits@10: {row[('test_hits@10', 'mean')]:.4f}")
            print(f"   Avg MR: {row[('test_mr', 'mean')]:.2f}")
            print(f"   Avg Training Time: {row[('training_time', 'mean')]/60:.1f} min")
    
    # Configuration comparison
    if len(configs_to_run) > 1:
        print(f"\n⚙️  PERFORMANCE BY CONFIGURATION:")
        print("-"*80)
        config_summary = df_results.groupby('config_name').agg({
            'test_mrr': ['mean', 'std', 'max'],
            'test_hits@10': ['mean', 'max'],
            'training_time': 'mean'
        }).round(4)
        
        for config_name in [c['name'] for c in configs_to_run]:
            if config_name in config_summary.index:
                row = config_summary.loc[config_name]
                config_desc = [c['description'] for c in configs_to_run if c['name'] == config_name][0]
                print(f"\n{config_name.upper()} ({config_desc}):")
                print(f"   Avg MRR: {row[('test_mrr', 'mean')]:.4f} ± {row[('test_mrr', 'std')]:.4f}")
                print(f"   Max MRR: {row[('test_mrr', 'max')]:.4f}")
                print(f"   Avg Hits@10: {row[('test_hits@10', 'mean')]:.4f}")
                print(f"   Avg Training Time: {row[('training_time', 'mean')]/60:.1f} min")
    
    # Detailed results table
    print(f"\n📋 DETAILED RESULTS TABLE:")
    print("-"*80)
    display_cols = ['config_name', 'pooling', 'test_mrr', 'test_hits@1', 'test_hits@10', 
                    'test_mr', 'hidden_dim', 'loss', 'training_time']
    df_display = df_results[display_cols].copy()
    df_display['test_mrr'] = df_display['test_mrr'].apply(lambda x: f"{x:.4f}")
    df_display['test_hits@1'] = df_display['test_hits@1'].apply(lambda x: f"{x:.4f}")
    df_display['test_hits@10'] = df_display['test_hits@10'].apply(lambda x: f"{x:.4f}")
    df_display['test_mr'] = df_display['test_mr'].apply(lambda x: f"{x:.2f}")
    df_display['training_time'] = df_display['training_time'].apply(lambda x: f"{x/60:.1f}min")
    
    print(df_display.to_string(index=False))
    
    # Save results
    results_csv_path = f"{OUTPUT_BASE_DIR}/fb15k237_all_results_summary.csv"
    df_results.to_csv(results_csv_path, index=False)
    print(f"\n✓ Saved detailed results to: {results_csv_path}")
    
    print("\n" + "="*80)
else:
    print("❌ No results available. Please run training experiments first.")

## 6. Visualization

Create comprehensive visualizations of experiment results.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    # Set style
    sns.set_style("whitegrid")
    
    # Create comprehensive visualization
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle('GWM-RNN Performance on FB15k-237 Dataset', fontsize=16, fontweight='bold')
    
    # 1. MRR comparison by pooling method
    ax1 = plt.subplot(2, 3, 1)
    pooling_data = df_results.groupby('pooling')['test_mrr'].apply(list)
    bp1 = ax1.boxplot([pooling_data[p] for p in POOLING_METHODS if p in pooling_data.index],
                       labels=[p.upper() for p in POOLING_METHODS if p in pooling_data.index],
                       patch_artist=True)
    for patch in bp1['boxes']:
        patch.set_facecolor('lightblue')
    ax1.set_ylabel('Test MRR', fontsize=11, fontweight='bold')
    ax1.set_title('MRR by Pooling Method', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 2. Hits@10 comparison
    ax2 = plt.subplot(2, 3, 2)
    pooling_h10 = df_results.groupby('pooling')['test_hits@10'].apply(list)
    bp2 = ax2.boxplot([pooling_h10[p] for p in POOLING_METHODS if p in pooling_h10.index],
                       labels=[p.upper() for p in POOLING_METHODS if p in pooling_h10.index],
                       patch_artist=True)
    for patch in bp2['boxes']:
        patch.set_facecolor('lightgreen')
    ax2.set_ylabel('Test Hits@10', fontsize=11, fontweight='bold')
    ax2.set_title('Hits@10 by Pooling Method', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # 3. Mean Rank comparison (lower is better)
    ax3 = plt.subplot(2, 3, 3)
    pooling_mr = df_results.groupby('pooling')['test_mr'].apply(list)
    bp3 = ax3.boxplot([pooling_mr[p] for p in POOLING_METHODS if p in pooling_mr.index],
                       labels=[p.upper() for p in POOLING_METHODS if p in pooling_mr.index],
                       patch_artist=True)
    for patch in bp3['boxes']:
        patch.set_facecolor('lightcoral')
    ax3.set_ylabel('Test Mean Rank', fontsize=11, fontweight='bold')
    ax3.set_title('Mean Rank by Pooling (Lower=Better)', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # 4. Scatter: MRR vs Training Time
    ax4 = plt.subplot(2, 3, 4)
    colors = {'last': 'blue', 'mean': 'green', 'max': 'red'}
    for pooling in POOLING_METHODS:
        mask = df_results['pooling'] == pooling
        ax4.scatter(df_results[mask]['training_time']/60, 
                   df_results[mask]['test_mrr'],
                   c=colors.get(pooling, 'gray'),
                   label=pooling.upper(),
                   s=100, alpha=0.6, edgecolors='black')
    ax4.set_xlabel('Training Time (minutes)', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Test MRR', fontsize=11, fontweight='bold')
    ax4.set_title('MRR vs Training Time', fontsize=12, fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Bar chart: Best MRR per pooling method
    ax5 = plt.subplot(2, 3, 5)
    best_per_pooling = df_results.groupby('pooling')['test_mrr'].max()
    bars = ax5.bar(range(len(best_per_pooling)), best_per_pooling.values, 
                   color=['blue', 'red', 'green'][:len(best_per_pooling)])
    ax5.set_xticks(range(len(best_per_pooling)))
    ax5.set_xticklabels([p.upper() for p in best_per_pooling.index])
    ax5.set_ylabel('Best Test MRR', fontsize=11, fontweight='bold')
    ax5.set_title('Best MRR per Pooling (FB15k-237)', fontsize=12, fontweight='bold')
    ax5.set_ylim([best_per_pooling.min() - 0.02, best_per_pooling.max() + 0.01])
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, best_per_pooling.values)):
        ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.4f}',
                ha='center', va='bottom', fontweight='bold', fontsize=9)
    ax5.grid(True, alpha=0.3, axis='y')
    
    # 6. Hits@K comparison
    ax6 = plt.subplot(2, 3, 6)
    hits_cols = ['test_hits@1', 'test_hits@3', 'test_hits@10']
    hits_means = df_results[hits_cols].mean()
    x = np.arange(len(hits_means))
    bars = ax6.bar(x, hits_means.values, color=['#ff9999', '#66b3ff', '#99ff99'])
    ax6.set_xticks(x)
    ax6.set_xticklabels(['Hits@1', 'Hits@3', 'Hits@10'])
    ax6.set_ylabel('Average Score', fontsize=11, fontweight='bold')
    ax6.set_title('Average Hits@K Scores', fontsize=12, fontweight='bold')
    
    # Add value labels
    for bar, val in zip(bars, hits_means.values):
        ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}\n({val*100:.1f}%)',
                ha='center', va='bottom', fontweight='bold', fontsize=9)
    ax6.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    # Save figure
    viz_path = f"{OUTPUT_BASE_DIR}/fb15k237_comprehensive_comparison.png"
    plt.savefig(viz_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Saved comprehensive visualization to: {viz_path}")
else:
    print("❌ No results to visualize.")

## 7. Best Model Analysis & Recommendations

Detailed analysis of the best performing model.

In [ ]:
if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    print("=" * 80)
    print("BEST MODEL ANALYSIS ON FB15k-237 DATASET")
    print("=" * 80)
    print()
    
    # Find best model
    best_idx = df_results['test_mrr'].idxmax()
    best_result = df_results.iloc[best_idx]
    
    print("🏆 BEST MODEL CONFIGURATION:")
    print(f"  • Configuration: {best_result['config_name'].upper()}")
    print(f"  • Pooling Method: {best_result['pooling'].upper()}")
    print(f"  • Hidden Dimension: {best_result['hidden_dim']}")
    print(f"  • LSTM Layers: {best_result['num_layers']}")
    print(f"  • Dropout: {best_result['dropout']:.2f}")
    print(f"  • Loss Function: {best_result['loss'].upper()}")
    print(f"  • Learning Rate: {best_result['learning_rate']}")
    print(f"  • Batch Size: {best_result['batch_size']}")
    print(f"  • Num Negatives: {best_result['num_negatives']}")
    print()
    
    print("📊 PERFORMANCE METRICS (Knowledge Graph Completion):")
    print(f"  • Test MRR:       {best_result['test_mrr']:.4f}")
    print(f"  • Test MR:        {best_result['test_mr']:.2f}")
    print(f"  • Test Hits@1:    {best_result['test_hits@1']:.4f} ({best_result['test_hits@1']*100:.2f}%)")
    print(f"  • Test Hits@3:    {best_result['test_hits@3']:.4f} ({best_result['test_hits@3']*100:.2f}%)")
    print(f"  • Test Hits@10:   {best_result['test_hits@10']:.4f} ({best_result['test_hits@10']*100:.2f}%)")
    print(f"  • Test Hits@50:   {best_result['test_hits@50']:.4f} ({best_result['test_hits@50']*100:.2f}%)")
    print()
    
    print("⏱️  TRAINING EFFICIENCY:")
    print(f"  • Training Time:  {best_result['training_time']:.1f}s ({best_result['training_time']/60:.1f} min)")
    print(f"  • Best Epoch:     {best_result['best_epoch']}")
    print(f"  • Best Val MRR:   {best_result['best_val_mrr']:.4f}")
    print()
    
    # Model size estimation
    hidden_dim = best_result['hidden_dim']
    num_layers = best_result['num_layers']
    input_dim = 384  # all-MiniLM-L6-v2 embedding dimension
    
    lstm_params_first = 4 * (input_dim * hidden_dim + hidden_dim * hidden_dim + hidden_dim)
    lstm_params_rest = (num_layers - 1) * 4 * (hidden_dim * hidden_dim + hidden_dim * hidden_dim + hidden_dim)
    projector_params = (input_dim * hidden_dim) + (hidden_dim * input_dim) + (hidden_dim * 2)
    total_params = lstm_params_first + lstm_params_rest + projector_params
    
    print("🔢 MODEL SIZE:")
    print(f"  • Total Parameters:     ~{total_params:,}")
    print(f"  • Model Size:           ~{total_params * 4 / (1024**2):.2f} MB (float32)")
    print()
    print("  📝 COMPARISON WITH BASELINES:")
    print(f"  • GWM-RNN (this model): {total_params:,} parameters")
    print(f"  • Typical LLM baseline:  3,000,000,000 parameters")
    print(f"  • Parameter Ratio:       1 : {3_000_000_000 / total_params:.0f}")
    print(f"  • This model is {3_000_000_000 / total_params:.0f}× more parameter-efficient!")
    print()
    
    # Ranking analysis
    print("📈 POOLING METHOD RANKING (by average MRR on FB15k-237):")
    pooling_ranking = df_results.groupby('pooling').agg({
        'test_mrr': 'mean',
        'test_hits@10': 'mean',
        'test_mr': 'mean',
        'training_time': 'mean'
    }).sort_values('test_mrr', ascending=False)
    
    for rank, (pooling, row) in enumerate(pooling_ranking.iterrows(), 1):
        print(f"  {rank}. {pooling.upper()}-pooling:")
        print(f"     • Avg MRR:      {row['test_mrr']:.4f}")
        print(f"     • Avg Hits@10:  {row['test_hits@10']:.4f}")
        print(f"     • Avg MR:       {row['test_mr']:.2f}")
        print(f"     • Avg Time:     {row['training_time']/60:.1f} min")
    print()
    
    # Recommendations
    print("💡 RECOMMENDATIONS:")
    print("  For PRODUCTION deployment on knowledge graph completion:")
    best_pooling = pooling_ranking.index[0]
    print(f"  • Use {best_pooling.upper()}-pooling for best MRR")
    print(f"  • Training time: ~{pooling_ranking.loc[best_pooling, 'training_time']/60:.1f} min (efficient!)")
    print(f"  • Expected MRR: ~{pooling_ranking.loc[best_pooling, 'test_mrr']:.4f}")
    print(f"  • Expected Hits@10: ~{pooling_ranking.loc[best_pooling, 'test_hits@10']*100:.1f}%")
    
    print()
    print("  For RESEARCH on knowledge graphs:")
    print("  • FB15k-237 is a challenging benchmark (MRR 0.25-0.35 is competitive)")
    print("  • Model trains efficiently (~60-90 min per config)")
    print("  • Lightweight architecture enables fast iteration")
    print("  • Consider hard negative mining for further improvements")
    print()
    print("=" * 80)
else:
    print("❌ No results available for best model analysis.")

## 8. Download Results

List of all output files and download instructions.

In [ ]:
print("=" * 80)
print("EXPERIMENT OUTPUT SUMMARY - FB15k-237 DATASET")
print("=" * 80)
print()
print(f"📁 Output directory: {OUTPUT_BASE_DIR}")
print()

# List experiment directories
if os.path.exists(OUTPUT_BASE_DIR):
    print("📂 EXPERIMENT DIRECTORIES:")
    for config_name in os.listdir(OUTPUT_BASE_DIR):
        config_path = os.path.join(OUTPUT_BASE_DIR, config_name)
        if os.path.isdir(config_path) and not config_name.startswith('.'):
            print(f"\n  • {config_name}/")
            for pooling_dir in os.listdir(config_path):
                pooling_path = os.path.join(config_path, pooling_dir)
                if os.path.isdir(pooling_path):
                    print(f"    └── {pooling_dir}/")
                    # List key files
                    files = os.listdir(pooling_path)
                    for file in sorted(files):
                        if file.endswith(('.pt', '.json', '.csv', '.png')):
                            file_path = os.path.join(pooling_path, file)
                            size_mb = os.path.getsize(file_path) / (1024 * 1024)
                            print(f"        ├── {file} ({size_mb:.2f} MB)")

    print()
    print("📊 SUMMARY FILES:")
    summary_files = [
        'fb15k237_all_results_summary.csv',
        'fb15k237_comprehensive_comparison.png'
    ]

    for file in summary_files:
        file_path = os.path.join(OUTPUT_BASE_DIR, file)
        if os.path.exists(file_path):
            size_mb = os.path.getsize(file_path) / (1024 * 1024)
            print(f"  ✓ {file} ({size_mb:.2f} MB)")
        else:
            print(f"  ✗ {file} (not found)")

    print()

    # Best model info
    if len(all_results) > 0:
        df_results = pd.DataFrame(all_results)
        best_idx = df_results['test_mrr'].idxmax()
        best_result = df_results.iloc[best_idx]
        
        print("🏆 BEST MODEL FILES:")
        best_model_dir = f"{OUTPUT_BASE_DIR}/{best_result['config_name']}/{best_result['pooling']}-pooling"
        print(f"  Location: {best_model_dir}/")
        print(f"  • Model checkpoint: best_model.pt")
        print(f"  • Training history: training_history.json")
        print(f"  • Test results: test_results.json")
        print(f"  • Configuration: config.json")
        print()

    print("💾 DOWNLOAD INSTRUCTIONS (Kaggle):")
    print("  1. Click the 'Save Version' button (top right)")
    print("  2. Select 'Save & Run All (Commit)'")
    print("  3. After completion, go to the 'Output' tab")
    print("  4. Download individual files or the entire output folder")
    print()
    print("  Alternative (via code):")
    print(f"  !zip -r fb15k237_experiments.zip {OUTPUT_BASE_DIR}")
    print()

    # Total size estimate
    total_size = 0
    for root, dirs, files in os.walk(OUTPUT_BASE_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            total_size += os.path.getsize(file_path)

    print(f"📦 Total output size: {total_size / (1024**2):.2f} MB")
    print()
    print("=" * 80)

    # Experiment summary
    if len(all_results) > 0:
        print()
        print("EXPERIMENT SUMMARY:")
        print(f"  • Total experiments run: {len(all_results)}")
        print(f"  • Pooling methods tested: {df_results['pooling'].nunique()}")
        print(f"  • Configurations tested: {df_results['config_name'].nunique()}")
        print(f"  • Best MRR: {df_results['test_mrr'].max():.4f}")
        print(f"  • Best Hits@10: {df_results['test_hits@10'].max():.4f} ({df_results['test_hits@10'].max()*100:.2f}%)")
        print(f"  • Total training time: {df_results['training_time'].sum()/60:.1f} min ({df_results['training_time'].sum()/3600:.1f} hours)")
        print("=" * 80)
else:
    print("Output directory not found yet. Run experiments first.")